In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Thu Sep 17 05:15:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate fastapi uvicorn python-multipart nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.4 MB/s eta 0:00:00


In [ ]:
BASE_MODEL = "/content/drive/.shortcut-targets-by-id/15Ajbs22aPdEYwjgP27KWlODonia4e0oO/huggingface_cache/models--Salesforce--blip2-opt-2.7b/snapshots/59a1ef6c1e5117b3f65523d1c6066825bcf315e3"

LORA_PATH = "/content/drive/MyDrive/SIH/satquery_blip2_lora"

import os

print("Base model exists:", os.path.exists(BASE_MODEL))
print("LoRA exists:", os.path.exists(LORA_PATH))

Base model exists: True
LoRA exists: True


In [ ]:
import torch

from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    BitsAndBytesConfig
)

from peft import PeftModel

processor = Blip2Processor.from_pretrained(BASE_MODEL)

quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = Blip2ForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    model,
    LORA_PATH
)

model.eval()

print("===================================")
print("SatQuery BLIP-2 + LoRA READY")
print("===================================")

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

SatQuery BLIP-2 + LoRA READY


In [ ]:
from google.colab import files

uploaded = files.upload()

from PIL import Image

image_path = next(iter(uploaded))

image = Image.open(image_path).convert("RGB")

question = "What is visible in this satellite image?"

prompt = f"Question: {question} Answer:"

inputs = processor(
    images=image,
    text=prompt,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

answer = processor.batch_decode(
    output,
    skip_special_tokens=True
)[0]

print("ANSWER:")
print(answer)

Saving Satellite image.jpg to Satellite image.jpg


ANSWER:
Question: What is visible in this satellite image? Answer: Cloudy and rain-tracked areas



In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import io
from PIL import Image
import torch

app = FastAPI(title="SatQuery Tool A")

@app.get("/health")
def health():
    return {
        "status": "ok",
        "tool": "TOOL_A_SINGLE_IMAGE_VQA",
        "model": "BLIP-2 OPT 2.7B + SatQuery LoRA"
    }

@app.post("/inference")
async def inference(
    image: UploadFile = File(...),
    question: str = Form(...)
):
    try:
        image_bytes = await image.read()
        pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        prompt = f"Question: {question} Answer:"
        inputs = processor(
            images=pil_image,
            text=prompt,
            return_tensors="pt"
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False
            )

        answer = processor.batch_decode(
            output,
            skip_special_tokens=True
        )[0]

        return {
            "status": "SUCCESS",
            "answer": answer,
            "model": "BLIP-2 OPT 2.7B + SatQuery LoRA"
        }

    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={
                "status": "ERROR",
                "error": str(e)
            }
        )

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
import subprocess
import time
import re

cloudflare = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_url = None

for _ in range(60):
    line = cloudflare.stdout.readline()

    if line:
        print(line, end="")

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:
            tunnel_url = match.group(0)
            break

    time.sleep(0.5)

print("\n==============================")
print("TOOL A TUNNEL URL:")
print(tunnel_url)
print("==============================")

2026-09-17T07:35:44Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-17T07:35:44Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-17T07:35:48Z INF +--------------------------------------------------------------------------------------------+
2026-09-17T07:35:48Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-17T07:35:48Z INF |  https://wagner-court-digital-brochures.trycloudflare.

In [ ]:
import requests

TOOL_A_URL = "https://gui-movers-ntsc-importantly.trycloudflare.com"

r = requests.get(TOOL_A_URL + "/health", timeout=30)

print("Status:", r.status_code)
print("Response:", r.text)

Status: 502
Response: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>trycloudflare.com | 502: Bad gateway</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-black-da